# CloudGuardian — prioritization model

Reproduces the Week 2 prioritization pipeline: loads Prowler CSPM findings,
scores each FAIL finding on `severity x exposure x blast_radius`, and writes
the ranked results into `consolidated_findings.db` (SQLite).

Replace `data/sample_findings.csv` with your real Prowler export
(192 findings) to reproduce the full run — the scoring logic is identical,
this notebook just wraps `build_db.py` with inline visibility into each step.


In [1]:
import csv
import sqlite3
from pathlib import Path
from collections import Counter

import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from build_db import (
    SEVERITY_WEIGHTS, EXPOSURE_RULES, BLAST_RADIUS_BY_RESOURCE_TYPE,
    score_exposure, score_blast_radius, load_and_score, write_db,
)

CSV_PATH = PROJECT_ROOT / "data/sample_findings.csv"
DB_PATH = PROJECT_ROOT / "db/consolidated_findings.db"
SCHEMA_PATH = PROJECT_ROOT / "db/schema.sql"


## 1. Load raw findings

In [2]:
with CSV_PATH.open() as f:
    raw_rows = list(csv.DictReader(f))

print(f"{len(raw_rows)} total findings loaded from {CSV_PATH.name}")
print(Counter(r["status"] for r in raw_rows))
print(Counter(r["severity"] for r in raw_rows))


10 total findings loaded from sample_findings.csv
Counter({'FAIL': 9, 'PASS': 1})
Counter({'critical': 4, 'high': 3, 'medium': 2, 'low': 1})


## 2. Scoring model

`priority_score = severity_score x exposure_score x blast_radius`

- **severity_score** — Prowler's own severity, weighted 1-10
- **exposure_score** — keyword-matched against the check_id (public access,
  open ingress, missing MFA, etc. score highest)
- **blast_radius** — fixed per resource type (IAM/RDS score highest — a
  compromised credential or database has the widest downstream impact)


In [3]:
print("Severity weights:", SEVERITY_WEIGHTS)
print()
print("Exposure rules (keyword -> score):")
for kw, score in EXPOSURE_RULES:
    print(f"  {kw:<35} {score}")
print()
print("Blast radius by resource type:", BLAST_RADIUS_BY_RESOURCE_TYPE)


Severity weights: {'critical': 10, 'high': 7, 'medium': 4, 'low': 2, 'informational': 1}

Exposure rules (keyword -> score):
  public_access                       10
  allow_ingress_from_internet         10
  no_public_access                    9
  no_mfa                              6
  overprivileged                      8
  encryption                          5
  versioning                          3
  logging                             6
  cloudtrail                          6

Blast radius by resource type: {'AwsIamUser': 9, 'AwsIamPolicy': 9, 'AwsRdsDbInstance': 9, 'AwsS3Bucket': 7, 'AwsEc2SecurityGroup': 7, 'AwsCloudTrailTrail': 8}


## 3. Score and rank

In [4]:
scored_rows = load_and_score(CSV_PATH)
fails = [r for r in scored_rows if r["status"].upper() == "FAIL"]

print(f"{len(fails)} FAIL findings ranked by priority\n")
print(f"{'Rank':<5}{'Score':<8}{'Sev':<10}{'MC':<7}{'Check':<45}")
for r in fails:
    print(f"{r['priority_rank']:<5}{r['priority_score']:<8.0f}{r['severity']:<10}"
          f"{(r['misconfig_id'] or '-'): <7}{r['check_id'][:44]:<45}")


9 FAIL findings ranked by priority

Rank Score   Sev       MC     Check                                        
1    900     critical  MC-06  rds_instance_no_public_access                
2    720     critical  MC-02  iam_policy_attached_is_overprivileged        
3    700     critical  MC-04  s3_bucket_public_access                      
4    700     critical  MC-05  ec2_securitygroup_allow_ingress_from_interne 
5    378     high      MC-03  iam_user_no_mfa                              
6    336     high      MC-07  cloudtrail_multi_region_enabled              
7    245     high      MC-01  s3_bucket_default_encryption                 
8    144     medium    -      iam_root_hardware_mfa_enabled                
9    84      medium    MC-08  s3_bucket_versioning_enabled                 


## 4. Persist to the consolidated findings database

In [5]:
write_db(scored_rows, DB_PATH, SCHEMA_PATH)
print(f"Wrote {len(scored_rows)} findings to {DB_PATH}")


Wrote 10 findings to /home/claude/cloudguardian-pipeline/db/consolidated_findings.db


## 5. Verify — query back from SQLite

In [6]:
conn = sqlite3.connect(DB_PATH)
top = conn.execute('''
    SELECT priority_rank, severity, misconfig_id, priority_score, title
    FROM findings
    WHERE status = "FAIL"
    ORDER BY priority_rank
    LIMIT 10
''').fetchall()

for row in top:
    print(row)

conn.close()


(1, 'critical', 'MC-06', 900.0, 'RDS instance is publicly accessible')
(2, 'critical', 'MC-02', 720.0, 'IAM policy grants wildcard admin-level actions')
(3, 'critical', 'MC-04', 700.0, 'S3 bucket allows public access')
(4, 'critical', 'MC-05', 700.0, 'Security group allows SSH from 0.0.0.0/0')
(5, 'high', 'MC-03', 378.0, 'IAM user does not have MFA enabled')
(6, 'high', 'MC-07', 336.0, 'CloudTrail logging is disabled')
(7, 'high', 'MC-01', 245.0, 'S3 bucket without default encryption')
(8, 'medium', '', 144.0, 'Root account does not have hardware MFA enabled')
(9, 'medium', 'MC-08', 84.0, 'S3 bucket versioning suspended')


## 6. Next step: remediation

For each top-ranked finding with a `misconfig_id`, the corresponding prompt
in `prompts/prompt_library.md` generates the remediation guidance, which
feeds either an auto-remediation Lambda (guardrailed, reversible changes)
or the human-approval queue (higher-risk changes like IAM/network access).
See `../lambdas/` for the remediation functions.
